# ACloudViewer Agent Integration Demo

This notebook demonstrates all features of the ACloudViewer agent integration:
- Format conversion (50+ formats)
- Point cloud processing (subsample, normals, registration)
- Mesh processing (simplify, smooth, subdivide, sample)
- Batch operations
- MCP tool calling

## Prerequisites
```bash
pip install 'cli-anything-acloudviewer[mcp,headless]'
```

In [ ]:
import subprocess
import json
import tempfile
from pathlib import Path

def run_cli(*args):
    """Run a CLI command and return parsed JSON output."""
    cmd = ["cli-anything-acloudviewer", "--json", "--mode", "headless"] + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
        return None
    return json.loads(result.stdout)

tmpdir = tempfile.mkdtemp(prefix="acv_demo_")
print(f"Working directory: {tmpdir}")

## 1. Backend Info

In [ ]:
info = run_cli("info")
print(json.dumps(info, indent=2))

## 2. List Supported Formats

In [ ]:
formats = run_cli("formats")
print(f"Point cloud formats ({len(formats['point_cloud'])}): {', '.join(formats['point_cloud'])}")
print(f"Mesh formats ({len(formats['mesh'])}): {', '.join(formats['mesh'])}")
print(f"Image formats ({len(formats['image'])}): {', '.join(formats['image'])}")
print(f"Total: {len(formats['all'])} formats")

## 3. Create Sample Data

In [ ]:
import cloudViewer as cv
import numpy as np

# Create a sample point cloud
cloud = cv.geometry.PointCloud()
pts = np.random.rand(10000, 3).astype(np.float64)
cloud.points = cv.utility.Vector3dVector(pts)
cloud.colors = cv.utility.Vector3dVector(np.random.rand(10000, 3))

cloud_path = f"{tmpdir}/sample_cloud.ply"
cv.io.write_point_cloud(cloud_path, cloud)
print(f"Created: {cloud_path} ({len(cloud.points)} points)")

# Create a sample mesh
mesh = cv.geometry.TriangleMesh.create_sphere(radius=1.0, resolution=20)
mesh.compute_vertex_normals()
mesh.paint_uniform_color([0.7, 0.3, 0.1])

mesh_path = f"{tmpdir}/sample_mesh.ply"
cv.io.write_triangle_mesh(mesh_path, mesh)
print(f"Created: {mesh_path} ({len(mesh.triangles)} triangles)")

## 4. Format Conversion

In [ ]:
# PLY -> PCD
result = run_cli("convert", cloud_path, f"{tmpdir}/cloud.pcd")
print("PLY -> PCD:", json.dumps(result, indent=2))

# PLY -> XYZ
result = run_cli("convert", cloud_path, f"{tmpdir}/cloud.xyz")
print("PLY -> XYZ:", json.dumps(result, indent=2))

# Mesh PLY -> OBJ
result = run_cli("convert", mesh_path, f"{tmpdir}/mesh.obj")
print("PLY -> OBJ:", json.dumps(result, indent=2))

# Mesh PLY -> STL
result = run_cli("convert", mesh_path, f"{tmpdir}/mesh.stl")
print("PLY -> STL:", json.dumps(result, indent=2))

## 5. Point Cloud Processing

In [ ]:
# Subsample
result = run_cli("process", "subsample", cloud_path,
                 "-o", f"{tmpdir}/subsampled.ply", "--voxel-size", "0.1")
print("Subsample:", json.dumps(result, indent=2))

# Compute normals
result = run_cli("process", "normals", cloud_path,
                 "-o", f"{tmpdir}/normals.ply")
print("Normals:", json.dumps(result, indent=2))

## 6. Mesh Processing

In [ ]:
# Mesh reconstruction from point cloud
result = run_cli("reconstruct", "mesh", f"{tmpdir}/normals.ply",
                 "-o", f"{tmpdir}/reconstructed.ply")
print("Reconstruction:", json.dumps(result, indent=2))

## 7. Batch Conversion

In [ ]:
# Create multiple files
batch_in = f"{tmpdir}/batch_input"
batch_out = f"{tmpdir}/batch_output"
Path(batch_in).mkdir(exist_ok=True)

for i in range(5):
    c = cv.geometry.PointCloud()
    c.points = cv.utility.Vector3dVector(np.random.rand(500, 3))
    cv.io.write_point_cloud(f"{batch_in}/cloud_{i}.ply", c)

result = run_cli("batch-convert", batch_in, batch_out, "--format", ".pcd")
print(f"Batch converted: {result['converted']} files")
print(f"Errors: {result['errors']}")

## 8. Session Management

In [ ]:
status = run_cli("session", "status")
print("Session:", json.dumps(status, indent=2))

## Summary

This demo showed:
- **50+ format** support for point clouds and meshes
- **Format conversion** including cross-type (cloud ↔ mesh)
- **Batch conversion** of entire directories
- **Point cloud processing** (subsample, normals)
- **Mesh reconstruction** from point clouds
- **Session management** (undo/redo/save)

### Additional capabilities (GUI mode):
- Scene tree management (list/add/remove/show/hide entities)
- Camera/view control (orientation, zoom, perspective)
- Screenshots
- Real-time processing (normals, subsample, crop)

### MCP Server (for AI agents):
```bash
python -m cli_anything.acloudviewer.mcp_server
```
Exposes 36 tools for OpenClaw, Cursor, Claude Code, etc.